# Module 2 - Topic 2: Overview of Popular LLMs (GPT, Claude, Gemini, LLaMA)
**Generative AI Fellowship — Beginner**

In this demo you will:
- Call GPT, Claude, and Gemini APIs from Python
- Send the same prompt to multiple models and compare outputs
- Observe differences in style, tone, and reasoning
- Understand how context window and pricing affect model selection


## Part 1 - Setup: Three Clients

In [ ]:
# install
# Run once to install all required libraries
# !pip install openai anthropic google-generativeai python-dotenv

In [ ]:
# setup
import os
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import google.generativeai as genai

load_dotenv()

# OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Anthropic client
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Google Gemini client
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
gemini_model = genai.GenerativeModel("gemini-1.5-flash")

print("All three clients ready.")

In [ ]:
# Helper functions to make calls cleaner
def ask_gpt(prompt, temperature=0.7):
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

def ask_claude(prompt, temperature=0.7):
    response = anthropic_client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

def ask_gemini(prompt):
    response = gemini_model.generate_content(prompt)
    return response.text

print("Helper functions ready.")

## Part 2 - Same Prompt, Three Models

In [ ]:
# same-prompt
prompt = "What are the three biggest challenges facing Nigerian entrepreneurs today? Be specific and practical."

print("=" * 60)
print("GPT-4o-mini:")
print("=" * 60)
print(ask_gpt(prompt))

print("\n" + "=" * 60)
print("Claude 3 Haiku:")
print("=" * 60)
print(ask_claude(prompt))

print("\n" + "=" * 60)
print("Gemini 1.5 Flash:")
print("=" * 60)
print(ask_gemini(prompt))

## Part 3 - Task-Specific Comparison

In [ ]:
# code-task
code_prompt = "Write a Python function called calculate_vat that takes a price in Naira and returns the price including 7.5% VAT. Include a docstring and handle the case where price is negative."

print("GPT-4o-mini:")
print("-" * 40)
print(ask_gpt(code_prompt, temperature=0))

print("\nClaude 3 Haiku:")
print("-" * 40)
print(ask_claude(code_prompt))

print("\nGemini 1.5 Flash:")
print("-" * 40)
print(ask_gemini(code_prompt))

In [ ]:
# writing-task
writing_prompt = "Write a two-sentence marketing tagline for a Nigerian ride-hailing app that targets professionals in Lagos. Make it memorable and locally relevant."

print("GPT-4o-mini:")
print(ask_gpt(writing_prompt))

print("\nClaude 3 Haiku:")
print(ask_claude(writing_prompt))

print("\nGemini 1.5 Flash:")
print(ask_gemini(writing_prompt))

## Part 4 - Context Window Demo

In [ ]:
# context-window
# Simulate a long document by repeating text
long_document = """
This is a sample Nigerian business report covering Q3 2024 performance.
Revenue increased by 23% year-on-year driven by growth in the fintech sector.
Key markets: Lagos (45%), Abuja (22%), Kano (18%), Port Harcourt (15%).
Challenges include naira depreciation, inflation at 32%, and infrastructure gaps.
Recommendations: focus on dollar-denominated revenue streams and expand to diaspora markets.
""" * 50  # Repeat to make it longer

summary_prompt = f"Summarise the following business report in three bullet points:\n\n{long_document}"

# Count approximate tokens (rough estimate: 1 token ≈ 4 characters)
approx_tokens = len(summary_prompt) // 4
print(f"Approximate prompt length: {approx_tokens} tokens")
print(f"Document length: {len(long_document)} characters")
print()

print("Sending to GPT-4o-mini...")
print(ask_gpt(summary_prompt))

## Part 5 - Pricing Reality Check

In [ ]:
# pricing-estimate
# Approximate pricing per 1M tokens (as of mid-2024)
pricing = {
    "GPT-4o-mini": {"input": 0.15, "output": 0.60},
    "GPT-4o": {"input": 5.00, "output": 15.00},
    "Claude 3 Haiku": {"input": 0.25, "output": 1.25},
    "Claude 3.5 Sonnet": {"input": 3.00, "output": 15.00},
    "Gemini 1.5 Flash": {"input": 0.075, "output": 0.30},
    "Gemini 1.5 Pro": {"input": 3.50, "output": 10.50},
}

# Scenario: customer service bot, 1000 conversations/day
# Average: 500 input tokens + 300 output tokens per conversation
daily_input_tokens = 1000 * 500
daily_output_tokens = 1000 * 300

print("Scenario: 1,000 customer service conversations per day")
print("Average: 500 input + 300 output tokens per conversation")
print("-" * 55)
print(f"{'Model':<25} {'Daily Cost':>12} {'Monthly Cost':>14}")
print("-" * 55)

for model, prices in pricing.items():
    daily_cost = (
        (daily_input_tokens / 1_000_000) * prices["input"] +
        (daily_output_tokens / 1_000_000) * prices["output"]
    )
    monthly_cost = daily_cost * 30
    print(f"{model:<25} ${daily_cost:>10.2f}  ${monthly_cost:>12.2f}")

## Part 6 - Nigerian Use Case Decision

In [ ]:
# use-case-decision
scenarios = [
    {
        "scenario": "A Lagos hospital wants an AI assistant to help doctors review patient notes",
        "best_model": "LLaMA 3 (self-hosted)",
        "reason": "Patient data cannot leave the hospital's infrastructure — privacy is non-negotiable"
    },
    {
        "scenario": "A law firm in Abuja wants to analyse 200-page contracts and extract key clauses",
        "best_model": "Claude 3.5 Sonnet",
        "reason": "200K token context window handles the full document; excellent writing quality for summaries"
    },
    {
        "scenario": "A fintech startup wants to build a coding assistant for their engineering team",
        "best_model": "GPT-4o",
        "reason": "Strongest code generation; reliable function calling for tool integration"
    },
    {
        "scenario": "A student wants to experiment with AI on a zero budget",
        "best_model": "Gemini 1.5 Flash (free tier) or LLaMA 3 (local)",
        "reason": "Free options that are still capable enough for learning and prototyping"
    }
]

for i, s in enumerate(scenarios, 1):
    print(f"Scenario {i}: {s['scenario']}")
    print(f"  Best model: {s['best_model']}")
    print(f"  Why: {s['reason']}")
    print()